In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
#datos de ingreso
dia_ivd = '20260401'
dia_fvd = '20260409'

In [3]:
# Ruta de la carpeta que contiene los archivos de desglosados zonal
ruta_carpeta = 'Z:/01 base_datos/41 Informe Diario CCZ/Blogger/Datos diarios  dia vencido/Perdidos'

# Fechas de inicio y fin para el filtro
fecha_inicio = f'{dia_ivd}'
fecha_fin = f'{dia_fvd}'

# Lista para almacenar los DataFrames
dataframes = []

# Recorrer todos los archivos en la carpeta
for nombre_archivo in os.listdir(ruta_carpeta):

    if nombre_archivo.endswith('_bitacora perdidos.csv'):

        # Extraer la fecha del nombre del archivo (YYYYMMDD)
        fecha_archivo = nombre_archivo[:8]

        # Verificar si la fecha está dentro del rango deseado
        if fecha_inicio <= fecha_archivo <= fecha_fin:

            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)

            # Leer el archivo
            df = pd.read_csv(ruta_archivo, encoding='latin')

            # Eliminar filas completamente vacías
            df = df.dropna(how='all')

            # Crear la columna Fecha a partir del nombre del archivo
            df['Fecha'] = pd.to_datetime(fecha_archivo, format='%Y%m%d')

            # Agregar a la lista
            dataframes.append(df)

# Verificar si se encontraron DataFrames
if dataframes:
    # Consolidar todos los DataFrames en uno solo
    perdidos = pd.concat(dataframes, ignore_index=True)
else:
    print("No se encontraron archivos para consolidar.")

# Verificar resultado
perdidos.head()


,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
0,2026-02-01,CE0590004,539,JONATHAN JOSE MONTIEL ARRIETA,510074.0,1.066521e+09,10:17:00,13:05:00,Falta de Movil,Movil llega tarde - Incorporación,...,"42,706","34,038","8,668",1.0,KAREN LORENA BARRERA SERRANO,2026-02-01 11:07:48,NO,Z50-4211,PATIO TINTAL,2026-02-01
1,2026-02-01,CE0590008,539,ALFONSO VIDAL FANDIÑO PEREZ,501335.0,1.611230e+07,13:53:00,16:39:00,Operador,Operador en otro servicio,...,"43,656",0,"43,656",1.0,WILSON ABEL TORRES FAJARDO,2026-02-01 14:25:05,NO,Z50-7122,PATIO TINTAL 2,2026-02-01
2,2026-02-01,CE0590008,539,ALFONSO VIDAL FANDIÑO PEREZ,501335.0,1.611230e+07,16:45:00,19:22:00,Operador,Operador en otro servicio,...,"42,706",0,"42,706",1.0,WILSON ABEL TORRES FAJARDO,2026-02-01 14:25:05,NO,Z50-7122,PATIO TINTAL 2,2026-02-01
3,2026-02-01,CE0590009,539,CRISTIAN EDUARDO FORERO CASTRO,505613.0,1.014257e+09,20:33:00,22:46:00,Operador,Operador en otro servicio,...,"43,656",0,"43,656",2.0,WILSON ABEL TORRES FAJARDO,2026-02-01 21:21:00,NO,CT-E900,PATIO TINTAL 2,2026-02-01
4,2026-02-01,CE0590010,539,OMAR ANDRES CORTES ROMERO,506905.0,1.013597e+09,21:21:00,23:12:00,Operatividad,Retraso por Congestión,...,"42,706",0,"42,706",2.0,WILSON ABEL TORRES FAJARDO,2026-02-01 22:02:43,NO,Z50-4182,PATIO TINTAL,2026-02-01


In [4]:
perdidos["FECHA"] = pd.to_datetime(perdidos["FECHA"])

In [5]:
fechas_filtrar = [
    "2026-02-07",
    "2026-02-08",
    "2026-02-14",
    "2026-02-15"
]

perdidos_filtrado = perdidos[
    perdidos["FECHA"].isin(pd.to_datetime(fechas_filtrar))
]

perdidos_filtrado.head()

,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
2259,2026-02-07,CE0580005,539,JOSE FABIAN ISAZA GIRALDO,506224.0,1.022978e+09,06:22:00,08:42:00,Falta de Movil,Movil llega tarde - Incorporación,...,"43,656","37,502","6,154",1.0,DORIS EDITH PATARROYO GORDO,2026-02-07 06:39:36,NO,Z50-7043,PATIO TINTAL 2,2026-02-07
2260,2026-02-07,CE0580008,539,JHON JAIRO RODRIGUEZ LLANOS,506133.0,8.023334e+07,04:36:00,06:38:00,Movil con Novedad,Varado en Vía,...,"43,656","26,213","17,443",1.0,ROSA MARIELA TELLEZ NAVARRO,2026-02-07 06:01:24,NO,Z50-7014,PATIO TINTAL 2,2026-02-07
2261,2026-02-07,CE0580011,539,GILMEST ANTONIO GARCIA MONTENEGRO,506663.0,7.997292e+07,17:11:00,19:58:00,Movil con Novedad,Vandalismo,...,"42,706","24,640","18,066",2.0,MAICOL ALEXANDER PITA BLANCO,2026-02-07 18:54:04,NO,Z50-7034,PATIO TINTAL 2,2026-02-07
2262,2026-02-07,CE0580011,539,DEIVID ARLEY SOCADAGUI PINZON,508410.0,1.000986e+09,07:17:00,09:49:00,Movil con Novedad,Varado en Vía,...,"43,656","12,572","31,084",1.0,DORIS EDITH PATARROYO GORDO,2026-02-07 09:54:22,NO,Z50-7036,PATIO TINTAL 2,2026-02-07
2263,2026-02-07,CE05B0002,12,ROBERTO CARLOS ARNEDO RINCON,506586.0,7.969433e+07,10:14:30,13:44:30,Movil con Novedad,Varado en Vía,...,"42,906","6,410","36,496",2.0,KAREN LORENA BARRERA SERRANO,2026-02-07 11:06:31,NO,Z50-2162,PATIO LA Y,2026-02-07


In [6]:
print(perdidos_filtrado['FECHA'].unique())

<DatetimeArray>
['2026-02-07 00:00:00', '2026-02-08 00:00:00', '2026-02-14 00:00:00',
 '2026-02-15 00:00:00']
Length: 4, dtype: datetime64[ns]


In [7]:
perdidos_filtrado["KMS PERDIDOS"] = (
    perdidos_filtrado["KMS PERDIDOS"]
    .astype(str)
    .str.replace(",", ".", regex=False)  # si usa coma decimal
    .str.replace(" ", "", regex=False)
    .str.strip()
)

perdidos_filtrado["KMS PERDIDOS"] = pd.to_numeric(
    perdidos_filtrado["KMS PERDIDOS"],
    errors="coerce"
)

perdidos_filtrado.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_30156\4039694383.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  perdidos_filtrado["KMS PERDIDOS"] = (
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_30156\4039694383.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  perdidos_filtrado["KMS PERDIDOS"] = pd.to_numeric(


,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
2259,2026-02-07,CE0580005,539,JOSE FABIAN ISAZA GIRALDO,506224.0,1.022978e+09,06:22:00,08:42:00,Falta de Movil,Movil llega tarde - Incorporación,...,"43,656","37,502",6.154,1.0,DORIS EDITH PATARROYO GORDO,2026-02-07 06:39:36,NO,Z50-7043,PATIO TINTAL 2,2026-02-07
2260,2026-02-07,CE0580008,539,JHON JAIRO RODRIGUEZ LLANOS,506133.0,8.023334e+07,04:36:00,06:38:00,Movil con Novedad,Varado en Vía,...,"43,656","26,213",17.443,1.0,ROSA MARIELA TELLEZ NAVARRO,2026-02-07 06:01:24,NO,Z50-7014,PATIO TINTAL 2,2026-02-07
2261,2026-02-07,CE0580011,539,GILMEST ANTONIO GARCIA MONTENEGRO,506663.0,7.997292e+07,17:11:00,19:58:00,Movil con Novedad,Vandalismo,...,"42,706","24,640",18.066,2.0,MAICOL ALEXANDER PITA BLANCO,2026-02-07 18:54:04,NO,Z50-7034,PATIO TINTAL 2,2026-02-07
2262,2026-02-07,CE0580011,539,DEIVID ARLEY SOCADAGUI PINZON,508410.0,1.000986e+09,07:17:00,09:49:00,Movil con Novedad,Varado en Vía,...,"43,656","12,572",31.084,1.0,DORIS EDITH PATARROYO GORDO,2026-02-07 09:54:22,NO,Z50-7036,PATIO TINTAL 2,2026-02-07
2263,2026-02-07,CE05B0002,12,ROBERTO CARLOS ARNEDO RINCON,506586.0,7.969433e+07,10:14:30,13:44:30,Movil con Novedad,Varado en Vía,...,"42,906","6,410",36.496,2.0,KAREN LORENA BARRERA SERRANO,2026-02-07 11:06:31,NO,Z50-2162,PATIO LA Y,2026-02-07


In [8]:
analisis_patio = (
    perdidos_filtrado.groupby("PATIO")
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count"),
        promedio_km_por_evento=("KMS PERDIDOS", "mean")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_patio.head()

,total_km_perdidos,eventos,promedio_km_por_evento
PATIO,,,
PATIO TINTAL,5704.241,144,39.612785
LA VERBENA_GM,4690.344,111,42.255351
PATIO TINTAL 2,1877.275,84,22.348512
PATIO LA Y,733.251,35,20.950029


In [9]:
analisis_patio_ruta = (
    perdidos_filtrado.groupby(["PATIO", "RUTA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_patio_ruta.head()

total_km_perdidos  eventos
PATIO         RUTA                             
PATIO TINTAL  DL219           2026.963       24
LA VERBENA_GM 576             1140.311       13
              577              966.845       19
PATIO TINTAL  806              793.291       28
LA VERBENA_GM BD237            746.420       25

In [10]:
analisis_motivo = (
    perdidos_filtrado.groupby("MOTIVO")
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_motivo

,total_km_perdidos,eventos
MOTIVO,,
Operador,7247.017,189
Movil con Novedad,3559.937,91
Operatividad,1259.569,45
Falta de Movil,938.588,49


In [11]:
analisis_causa = (
    perdidos_filtrado.groupby(["MOTIVO", "CAUSA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_causa.head(20)

total_km_perdidos  \
MOTIVO            CAUSA                                                     
Operador          Operador no se presenta                        5774.611   
Movil con Novedad Varado en Vía                                  1748.404   
                  Falla Sirci                                    1450.684   
Operador          Operador en otro servicio                      1084.276   
Operatividad      Desvio autorizado                               935.431   
Falta de Movil    Sin Movil para retomar                          677.210   
Movil con Novedad Accidente                                       272.067   
Operatividad      Retraso por Congestión                          220.529   
Falta de Movil    Movil llega tarde - Incorporación               148.792   
Operador          Operador con novedad en vía                     146.430   
Falta de Movil    Movil no despachado                             101.682   
Operador          Operador llega tarde                             92.104   
Movil con Novedad Vandalismo                                       88.782   
Operador          Operador se niega a salir                        61.727   
Operatividad      Baja demanda                                     53.082   
Operador          Operador llega tarde - Incorporación             44.502   
                  Operador se equivoca de PIR                      43.367   
Operatividad      Desvio No autorizado                             35.683   
                  Perdida por aseo                                 14.844   
Falta de Movil    Movil llega tarde                                10.904   

                                                        eventos  
MOTIVO            CAUSA                                          
Operador          Operador no se presenta                   146  
Movil con Novedad Varado en Vía                              49  
                  Falla Sirci                                25  
Operador          Operador en otro servicio                  27  
Operatividad      Desvio autorizado                           2  
Falta de Movil    Sin Movil para retomar                     22  
Movil con Novedad Accidente                                  12  
Operatividad      Retraso por Congestión                     22  
Falta de Movil    Movil llega tarde - Incorporación          23  
Operador          Operador con novedad en vía                 5  
Falta de Movil    Movil no despachado                         3  
Operador          Operador llega tarde                        3  
Movil con Novedad Vandalismo                                  5  
Operador          Operador se niega a salir                   1  
Operatividad      Baja demanda                               10  
Operador          Operador llega tarde - Incorporación        6  
                  Operador se equivoca de PIR                 1  
Operatividad      Desvio No autorizado                       10  
                  Perdida por aseo                            1  
Falta de Movil    Movil llega tarde                           1

In [12]:
resumen_ejecutivo = (
    perdidos_filtrado.groupby(["PATIO", "MOTIVO"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .groupby(level=0)
    .apply(lambda x: x.sort_values("total_km_perdidos", ascending=False).head(3))
)

resumen_ejecutivo

total_km_perdidos
PATIO          PATIO          MOTIVO                              
LA VERBENA_GM  LA VERBENA_GM  Operador                    2280.104
                              Movil con Novedad           1132.777
                              Operatividad                 967.915
PATIO LA Y     PATIO LA Y     Movil con Novedad            358.813
                              Operador                     275.142
                              Operatividad                  68.623
PATIO TINTAL   PATIO TINTAL   Operador                    3638.087
                              Movil con Novedad           1691.078
                              Falta de Movil               329.468
PATIO TINTAL 2 PATIO TINTAL 2 Operador                    1053.684
                              Movil con Novedad            377.269
                              Falta de Movil               268.899

Ruta en particular

In [13]:
kb309_4dias = perdidos_filtrado[
    (perdidos_filtrado["RUTA"] == "KB309") &
    (perdidos_filtrado["FECHA"].dt.month == 2) &
    (perdidos_filtrado["FECHA"].dt.day.isin([7, 8, 14, 15]))
]

kb309_4dias

,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
2304,2026-02-07,CE09D0008,KB309,JOHN EDISSON RAMOS MORALES,510367.0,1.016031e+09,16:30:30,21:29:30,Movil con Novedad,Varado en Vía,...,"74,740","71,041",3.699,1.0,ESTEBAN CAMILO MEDINA,2026-02-07 21:29:57,NO,Z50-4007,PATIO LA Y,2026-02-07
2414,2026-02-08,CE09E0006,KB309,KEVIN NICOLAS LANCHEROS RAMIREZ,510014.0,1.016594e+09,15:58:00,20:27:00,Movil con Novedad,Falla Sirci,...,"74,740","35,354",39.386,1.0,ANGELA MIREYA RIAÑO PASITO,2026-02-08 18:24:43,NO,Z50-4005,PATIO LA Y,2026-02-08
4277,2026-02-14,CE09D0018,KB309,JAIRO ALONSO MELO MORENO,510167.0,1.012378e+09,08:15:30,14:19:30,Movil con Novedad,Varado en Vía,...,"74,740","36,017",38.723,1.0,WILLIAN HUMBERTO MUNAR GONZALEZ,2026-02-14 11:09:31,NO,Z52-4002,PATIO LA Y,2026-02-14
4365,2026-02-15,CE09E0006,KB309,JUAN PABLO DOZA RATIVA,510113.0,1.020805e+09,11:07:00,15:51:00,Operatividad,Desvio No autorizado,...,"77,622","75,974",1.648,1.0,ANGELA MIREYA RIAÑO PASITO,2026-02-15 12:56:18,NO,Z50-4024,PATIO LA Y,2026-02-15


In [14]:
resumen_dia = (
    kb309_4dias.groupby("FECHA")
    .agg(
        eventos=("KMS PERDIDOS", "count"),
        total_km=("KMS PERDIDOS", "sum"),
        promedio_km=("KMS PERDIDOS", "mean")
    )
)

resumen_dia

,eventos,total_km,promedio_km
FECHA,,,
2026-02-07,1,3.699,3.699
2026-02-08,1,39.386,39.386
2026-02-14,1,38.723,38.723
2026-02-15,1,1.648,1.648


In [15]:
motivos_kb309 = (
    kb309_4dias.groupby("MOTIVO")
    .agg(
        eventos=("KMS PERDIDOS", "count"),
        total_km=("KMS PERDIDOS", "sum")
    )
    .sort_values("total_km", ascending=False)
)

motivos_kb309

,eventos,total_km
MOTIVO,,
Movil con Novedad,3,81.808
Operatividad,1,1.648


In [16]:
causa_kb309 = (
    kb309_4dias.groupby(["MOTIVO", "CAUSA"])
    .agg(
        eventos=("KMS PERDIDOS", "count"),
        total_km=("KMS PERDIDOS", "sum")
    )
    .sort_values("total_km", ascending=False)
)

causa_kb309

eventos  total_km
MOTIVO            CAUSA                                  
Movil con Novedad Varado en Vía               2    42.422
                  Falla Sirci                 1    39.386
Operatividad      Desvio No autorizado        1     1.648

In [17]:
perdidos_filtrado.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Análisis_perdidos/perdidos_reporte_bitácora.csv', index=False, sep=';')